Historical plotting notebook. Start Jupyter from the repository root; use the experiment READMEs and command-line runners to generate inputs first. Cells retain exploratory plotting variants; Run All is not the reproduction entry point.

In [ ]:
from pathlib import Path
import os, sys
while not Path("pyproject.toml").exists():
    if Path.cwd() == Path.cwd().parent: raise RuntimeError("Start Jupyter inside this repository")
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap


In [ ]:

# Create output directory
output_dir = "exp4_adv_samples_analysis"
os.makedirs(output_dir, exist_ok=True)

In [ ]:

# Set the root directory for adversarial samples
root_dir = "~/Desktop/Research/My Projects/neural_expansion/adv_samples"
root_dir = os.path.expanduser(root_dir)  # Expand the ~ character

In [ ]:
# Set consistent font sizes and style
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 14
plt.rcParams['figure.titlesize'] = 18

In [ ]:

# Function to load all GI CSV files
def load_gi_data(root_dir):
    results = {
        "dataset": [],
        "model": [],
        "attack": [],
        "classification": [],
        "percent_gi_above_1": [],
        "sample_count": []
    }
    
    # Walk through all directories and find CSV files
    for dataset_dir in os.listdir(root_dir):
        dataset_path = os.path.join(root_dir, dataset_dir)
        if not os.path.isdir(dataset_path):
            continue
            
        for model_attack_dir in os.listdir(dataset_path):
            model_attack_path = os.path.join(dataset_path, model_attack_dir)
            if not os.path.isdir(model_attack_path):
                continue
                
            # Parse model and attack from directory name
            parts = model_attack_dir.split('_')
            if len(parts) < 2:
                continue
                
            # The last part is the attack, everything before is the model
            attack = parts[-1]
            model = '_'.join(parts[:-1])
            
            # Check for correct_gi.csv and wrong_gi.csv
            for classification in ["correct", "wrong"]:
                csv_path = os.path.join(model_attack_path, f"{classification}_gi.csv")
                if os.path.exists(csv_path):
                    try:
                        df = pd.read_csv(csv_path)
                        avg_gi_percent = df["percent_gi_above_1"].mean()
                        
                        results["dataset"].append(dataset_dir)
                        results["model"].append(model)
                        results["attack"].append(attack)
                        results["classification"].append(classification)
                        results["percent_gi_above_1"].append(avg_gi_percent)
                        results["sample_count"].append(len(df))
                    except Exception as e:
                        print(f"Error reading {csv_path}: {e}")
    
    return pd.DataFrame(results)

# Load all GI data
gi_data = load_gi_data(root_dir)

# Display the loaded data
print("Loaded GI data:")
print(gi_data)

# Add a combined dataset-model field
gi_data['dataset_model'] = gi_data['dataset'] + '-' + gi_data['model']

# Define the desired order of datasets-models
dataset_model_order = ['mnist-mlp', 'cifar10-resnet20', 'imagenette-vitbase']

# Define the desired display names 
dataset_model_display = {
    'mnist-mlp': 'MNIST-MLP',
    'cifar10-resnet20': 'CIFAR10-ResNet20', 
    'imagenette-vitbase': 'Imagenette-ViTB'
}

# Replace dataset_model names with display names
gi_data['dataset_model_display'] = gi_data['dataset_model'].map(dataset_model_display)

# Create attack-classification combined field 
gi_data['attack_classification'] = gi_data['attack'] + '-' + gi_data['classification']

# Define display names for attacks and classifications
attack_classification_display = {
    'cw-correct': 'CW-Correct',
    'cw-wrong': 'CW-Wrong',
    'ifgsm-correct': 'IFGSM-Correct',
    'ifgsm-wrong': 'IFGSM-Wrong',
    'pgd-correct': 'PGD-Correct',
    'pgd-wrong': 'PGD-Wrong'
}

# Replace attack_classification with display names
gi_data['attack_classification_display'] = gi_data['attack_classification'].map(attack_classification_display)

# ------------------- Visualization: Heatmap -------------------
plt.figure(figsize=(14, 3), facecolor='white')

# Create pivot table for heatmap
try:
    # Create a pivot table
    heatmap_data = pd.pivot_table(
        gi_data,
        values='percent_gi_above_1',
        index=['dataset_model_display'],
        columns=['attack_classification_display'],
        aggfunc='mean'
    )
    
    # Reorder the index based on our desired order
    index_order = [dataset_model_display[dm] for dm in dataset_model_order]
    heatmap_data = heatmap_data.reindex(index_order)
    
    # Set the column order
    column_order = ['CW-Correct', 'CW-Wrong', 'IFGSM-Correct', 'IFGSM-Wrong', 'PGD-Correct', 'PGD-Wrong']
    heatmap_data = heatmap_data.reindex(columns=column_order)
    
    # Create a mask for NaN values
    mask = heatmap_data.isna()
    
    # Create a custom colormap with red to green (low to high)
    # where red is for lower values and green for higher values
    cmap = LinearSegmentedColormap.from_list("custom_RdYlGn", 
                                          [(0, '#e74c3c'),    # Red for low values
                                           (0.5, '#f1c40f'),  # Yellow for middle values
                                           (1, '#2ecc71')])   # Green for high values
    
    # Plot the heatmap
    ax = sns.heatmap(
        heatmap_data, 
        annot=True,  
        fmt=".1f", 
        cmap=cmap,
        linewidths=0.5,
        cbar_kws={'label': '% with GI > 1'},
        mask=mask,  # Mask NaN values
        annot_kws={"size": 12}  # Set font size for annotations
    )
    
    # Fill NaN cells with light blue color and a dash
    for i, idx in enumerate(heatmap_data.index):
        for j, col in enumerate(heatmap_data.columns):
            if pd.isna(heatmap_data.loc[idx, col]):
                # Add a light blue rectangle
                rect = plt.Rectangle((j, i), 1, 1, fill=True, color='#add8e6', alpha=0.7)
                ax.add_patch(rect)
                # Add a dash
                ax.text(j + 0.5, i + 0.5, "-", ha='center', va='center', color='black', fontsize=12, fontweight='bold')
    
    # Set the titles and labels
    plt.title("Comprehensive GI Analysis Across Models, Datasets, and Attack Types", fontsize=16)
    plt.xlabel("Attack-Classification", fontsize=14)
    plt.ylabel("Dataset-Model", fontsize=14)
    
    plt.tight_layout()
    
    # Save to the output directory with both PDF and PNG formats
    heatmap_output_path_png = os.path.join(output_dir, "gi_heatmap_analysis.png")
    plt.savefig(heatmap_output_path_png, dpi=300, bbox_inches="tight", facecolor='white')
    
    heatmap_output_path_pdf = os.path.join(output_dir, "gi_heatmap_analysis.pdf")
    plt.savefig(heatmap_output_path_pdf, bbox_inches="tight", facecolor='white')
    
    print(f"Heatmap analysis figures saved to:")
    print(f"  - {heatmap_output_path_png}")
    print(f"  - {heatmap_output_path_pdf}")
except Exception as e:
    print(f"Couldn't create heatmap: {e}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set the root directory for adversarial samples
root_dir = "~/Desktop/Research/My Projects/neural_expansion/adv_samples"
root_dir = os.path.expanduser(root_dir)  # Expand the ~ character

# Function to load all GI CSV files
def load_gi_data(root_dir):
    results = {
        "dataset": [],
        "model": [],
        "attack": [],
        "classification": [],
        "percent_gi_above_1": [],
        "sample_count": []
    }
    
    # Walk through all directories and find CSV files
    for dataset_dir in os.listdir(root_dir):
        dataset_path = os.path.join(root_dir, dataset_dir)
        if not os.path.isdir(dataset_path):
            continue
            
        for model_attack_dir in os.listdir(dataset_path):
            model_attack_path = os.path.join(dataset_path, model_attack_dir)
            if not os.path.isdir(model_attack_path):
                continue
                
            # Parse model and attack from directory name
            parts = model_attack_dir.split('_')
            if len(parts) < 2:
                continue
                
            # The last part is the attack, everything before is the model
            attack = parts[-1]
            model = '_'.join(parts[:-1])
            
            # Check for correct_gi.csv and wrong_gi.csv
            for classification in ["correct", "wrong"]:
                csv_path = os.path.join(model_attack_path, f"{classification}_gi.csv")
                if os.path.exists(csv_path):
                    try:
                        df = pd.read_csv(csv_path)
                        avg_gi_percent = df["percent_gi_above_1"].mean()
                        
                        results["dataset"].append(dataset_dir)
                        results["model"].append(model)
                        results["attack"].append(attack)
                        results["classification"].append(classification)
                        results["percent_gi_above_1"].append(avg_gi_percent)
                        results["sample_count"].append(len(df))
                    except Exception as e:
                        print(f"Error reading {csv_path}: {e}")
    
    return pd.DataFrame(results)

# Load all GI data
gi_data = load_gi_data(root_dir)

# Display the loaded data
print("Loaded GI data:")
print(gi_data)

# ------------------- Visualization 1: Complete Overview -------------------
plt.figure(figsize=(16, 8))
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)

# Create a palette with distinct colors
palette = {"correct": "#2ecc71", "wrong": "#e74c3c"}

# Create a combined label for better visualization
gi_data['combined_label'] = gi_data['model'] + '-' + gi_data['dataset'] + '-' + gi_data['attack']

# Sort data
gi_data_sorted = gi_data.sort_values(['dataset', 'model', 'attack', 'classification'])

# Plot using factorplot (now called catplot) for better control
g = sns.catplot(
    data=gi_data_sorted,
    kind="bar",
    x="combined_label",
    y="percent_gi_above_1",
    hue="classification",
    palette=palette,
    height=6,
    aspect=2.5,
    legend_out=False
)

# Customize the plot
g.set_xticklabels(rotation=45, ha="right")
g.set(xlabel="", ylabel="% of Examples with GI > 1")
g.fig.suptitle("Generalization Interval Analysis Across Models, Datasets and Attack Types", fontsize=16, y=1.02)

# Extract the axes for annotation
ax = g.axes[0, 0]

# Add sample count annotations safely
for i, row in gi_data_sorted.reset_index().iterrows():
    # Find the corresponding bar
    bars = [p for p in ax.patches if abs(p.get_x() + p.get_width()/2 - i % len(gi_data_sorted['combined_label'].unique())) < 0.2]
    for bar in bars:
        height = bar.get_height()
        # Check if this is the right classification by color
        if ((bar.get_facecolor()[0] > 0.5 and row['classification'] == 'correct') or 
            (bar.get_facecolor()[0] < 0.5 and row['classification'] == 'wrong')):
            # Add the count annotation
            ax.text(bar.get_x() + bar.get_width()/2, height + 1,
                    f"n={row['sample_count']}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()

# Save the figure
output_path = os.path.join(root_dir, "gi_adversarial_analysis.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to {output_path}")

# ------------------- Visualization 2: By Attack Type -------------------
plt.figure(figsize=(10, 6))

# Aggregate data by attack type and classification
attack_summary = gi_data.groupby(['attack', 'classification']).agg({
    'percent_gi_above_1': 'mean',
    'sample_count': 'sum'
}).reset_index()

# Plot
attack_plot = sns.barplot(
    data=attack_summary,
    x='attack',
    y='percent_gi_above_1',
    hue='classification',
    palette=palette,
    errorbar=None
)

# Add sample counts directly from the aggregated data
for i, row in attack_summary.iterrows():
    attack_plot.text(
        i % len(attack_summary['attack'].unique()),
        row['percent_gi_above_1'] + 1,
        f"n={int(row['sample_count'])}",
        ha='center',
        va='bottom',
        fontsize=10
    )

attack_plot.set_xlabel("Attack Type", fontsize=12)
attack_plot.set_ylabel("Average % with GI > 1", fontsize=12)
attack_plot.set_title("Impact of Attack Type on Generalization Intervals", fontsize=14)
plt.legend(title="Classification")
plt.tight_layout()

attack_output_path = os.path.join(root_dir, "gi_attack_analysis.png")
plt.savefig(attack_output_path, dpi=300, bbox_inches="tight")
print(f"Attack analysis figure saved to {attack_output_path}")

# ------------------- Visualization 3: By Dataset -------------------
plt.figure(figsize=(10, 6))

# Aggregate data by dataset and classification
dataset_summary = gi_data.groupby(['dataset', 'classification']).agg({
    'percent_gi_above_1': 'mean',
    'sample_count': 'sum'
}).reset_index()

# Plot
dataset_plot = sns.barplot(
    data=dataset_summary,
    x='dataset',
    y='percent_gi_above_1',
    hue='classification',
    palette=palette,
    errorbar=None
)

# Add sample counts
for i, row in dataset_summary.iterrows():
    dataset_plot.text(
        i % len(dataset_summary['dataset'].unique()),
        row['percent_gi_above_1'] + 1,
        f"n={int(row['sample_count'])}",
        ha='center',
        va='bottom',
        fontsize=10
    )

dataset_plot.set_xlabel("Dataset", fontsize=12)
dataset_plot.set_ylabel("Average % with GI > 1", fontsize=12)
dataset_plot.set_title("GI Analysis by Dataset", fontsize=14)
plt.legend(title="Classification")
plt.tight_layout()

dataset_output_path = os.path.join(root_dir, "gi_dataset_analysis.png")
plt.savefig(dataset_output_path, dpi=300, bbox_inches="tight")
print(f"Dataset analysis figure saved to {dataset_output_path}")

# ------------------- Visualization 4: Heatmap -------------------
# Only create the heatmap if we have enough data
if len(gi_data) >= 4:
    plt.figure(figsize=(12, 4))
    
    # Create pivot table for heatmap
    try:
        heatmap_data = pd.pivot_table(
            gi_data,
            values='percent_gi_above_1',
            index=['dataset', 'model'],
            columns=['attack', 'classification'],
            aggfunc='mean'
        )
        
        # Plot the heatmap
        ax = sns.heatmap(
            heatmap_data, 
            annot=True, 
            fmt=".1f", 
            cmap="RdYlGn",
            linewidths=0.5,
            cbar_kws={'label': '% with GI > 1'}
        )
        
        plt.title("Comprehensive GI Analysis Across Models, Datasets, and Attack Types", fontsize=14)
        plt.tight_layout()
        
        heatmap_output_path = os.path.join(root_dir, "gi_heatmap_analysis.png")
        plt.savefig(heatmap_output_path, dpi=300, bbox_inches="tight")
        print(f"Heatmap analysis figure saved to {heatmap_output_path}")
    except Exception as e:
        print(f"Couldn't create heatmap: {e}")

# ------------------- Visualization 5: Combined Comparison -------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Filter data
correct_data = gi_data[gi_data['classification'] == 'correct']
wrong_data = gi_data[gi_data['classification'] == 'wrong']

# Create grouped data
correct_by_attack = correct_data.groupby('attack')['percent_gi_above_1'].mean().reset_index()
wrong_by_attack = wrong_data.groupby('attack')['percent_gi_above_1'].mean().reset_index()

# Sample counts
correct_counts = correct_data.groupby('attack')['sample_count'].sum().reset_index()
wrong_counts = wrong_data.groupby('attack')['sample_count'].sum().reset_index()

# Create the plots
axes[0].bar(correct_by_attack['attack'], correct_by_attack['percent_gi_above_1'], color='#2ecc71', alpha=0.8)
axes[0].set_title("Correctly Classified Samples", fontsize=12)
axes[0].set_ylim(0, 100)
axes[0].set_ylabel("Average % with GI > 1", fontsize=10)

axes[1].bar(wrong_by_attack['attack'], wrong_by_attack['percent_gi_above_1'], color='#e74c3c', alpha=0.8)
axes[1].set_title("Misclassified Samples", fontsize=12)
axes[1].set_ylim(0, 100)

# Add counts
for i, (attack, value) in enumerate(zip(correct_by_attack['attack'], correct_by_attack['percent_gi_above_1'])):
    count = correct_counts[correct_counts['attack'] == attack]['sample_count'].values[0]
    axes[0].text(i, value + 2, f"n={count}", ha='center')

for i, (attack, value) in enumerate(zip(wrong_by_attack['attack'], wrong_by_attack['percent_gi_above_1'])):
    count = wrong_counts[wrong_counts['attack'] == attack]['sample_count'].values[0]
    axes[1].text(i, value + 2, f"n={count}", ha='center')

plt.suptitle("Comparison of GI > 1 Percentages by Classification Result", fontsize=14)
plt.tight_layout()

comparison_output_path = os.path.join(root_dir, "gi_comparison_analysis.png")
plt.savefig(comparison_output_path, dpi=300, bbox_inches="tight")
print(f"Comparison analysis figure saved to {comparison_output_path}")